In [1]:
import os
print(os.cpu_count())
print(os.getcwd())


20
C:\Users\WWWZX\AI_Class_Work\Transformers


In [2]:
import torch
print(torch.__version__)
# print(torch.cuda.is_available())

2.6.0+cu126


In [3]:
import gc
import torch

print(gc.collect())
torch.cuda.empty_cache()


20


In [4]:
import random
random.seed(69)

#NumPy random seed
import numpy as np
np.random.seed(69)

#PyTorch random seed
import torch
torch.manual_seed(69)
torch.cuda.manual_seed(69)
torch.cuda.manual_seed_all(69)  # If using multi-GPU
torch.cuda.empty_cache()
torch.backends.cudnn.deterministic = False  # For deterministic results
torch.backends.cudnn.benchmark = True  # Disabling to ensure deterministic algorithm
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision('medium')  # Optimize matmul precision

In [5]:
import torch
print("CUDA Available:", torch.cuda.is_available())
print("CUDA Version:", torch.version.cuda)
print("Torch CUDA Version:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU found")


CUDA Available: True
CUDA Version: 12.6
Torch CUDA Version: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
import torch

# Check if CUDA is available
gpu_available = torch.cuda.is_available()
print("CUDA Available:", gpu_available)

if gpu_available:
    # Get the number of GPUs available
    num_gpus = torch.cuda.device_count()
    print("Number of GPUs available:", num_gpus)

CUDA Available: True
Number of GPUs available: 1


In [7]:
from pathlib import Path
import random
import gc
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

import lightning.pytorch as pl
from lightning.pytorch import Trainer
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint,RichProgressBar,TQDMProgressBar,BatchSizeFinder
from lightning.pytorch.loggers import TensorBoardLogger
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import MAE, SMAPE, PoissonLoss, QuantileLoss, RMSE
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters
from pytorch_forecasting.data.encoders import NaNLabelEncoder
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
import seaborn as sns
import matplotlib.pyplot as plt
# from lightning.pytorch.tuner.tuning import Tuner
torch.serialization.add_safe_globals([TimeSeriesDataSet, GroupNormalizer])

In [8]:
file_path = "even_further_processed_sales_data_optimized_dtypes.parquet"
data = pd.read_parquet(file_path)


In [9]:
type(data)

pandas.core.frame.DataFrame

In [10]:
data.head()

,id,date,store_nbr,item_nbr,unit_sales,is_onpromotion,is_holiday,transactions,day,month,weekday,year,log_unit_sales,time_idx
0,0,2013-01-01,25,103665,7.0,0,1,0.092116,0.0,0.0,0.166667,0.0,2.079442,0
1,1,2013-01-01,25,105574,1.0,0,1,0.092116,0.0,0.0,0.166667,0.0,0.693147,0
2,2,2013-01-01,25,105575,2.0,0,1,0.092116,0.0,0.0,0.166667,0.0,1.098612,0
3,3,2013-01-01,25,108079,1.0,0,1,0.092116,0.0,0.0,0.166667,0.0,0.693147,0
4,4,2013-01-01,25,108701,1.0,0,1,0.092116,0.0,0.0,0.166667,0.0,0.693147,0


In [11]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 126307884 entries, 0 to 126310788
Data columns (total 14 columns):
 #   Column          Dtype         
---  ------          -----         
 0   id              uint32        
 1   date            datetime64[ms]
 2   store_nbr       category      
 3   item_nbr        category      
 4   unit_sales      float32       
 5   is_onpromotion  category      
 6   is_holiday      category      
 7   transactions    float64       
 8   day             float64       
 9   month           float64       
 10  weekday         float64       
 11  year            float64       
 12  log_unit_sales  float32       
 13  time_idx        int32         
dtypes: category(4), datetime64[ms](1), float32(2), float64(5), int32(1), uint32(1)
memory usage: 9.1 GB


In [12]:
data.columns

Index(['id', 'date', 'store_nbr', 'item_nbr', 'unit_sales', 'is_onpromotion',
       'is_holiday', 'transactions', 'day', 'month', 'weekday', 'year',
       'log_unit_sales', 'time_idx'],
      dtype='object')

In [13]:
data.dtypes

id                        uint32
date              datetime64[ms]
store_nbr               category
item_nbr                category
unit_sales               float32
is_onpromotion          category
is_holiday              category
transactions             float64
day                      float64
month                    float64
weekday                  float64
year                     float64
log_unit_sales           float32
time_idx                   int32
dtype: object

In [14]:
print("Checking for NaNs:", data.isna().sum().sum())  # Count total 
print("Checking for Infs:", np.isinf(data.select_dtypes(include=[np.number])).sum().sum())  # Count total Infs

Checking for NaNs: 0
Checking for Infs: 0


In [15]:
data = data.dropna().replace([np.inf, -np.inf], np.nan).dropna()

In [16]:
print(data.memory_usage(deep=True))
print(data.memory_usage(deep=True).sum() / (1024 ** 3), "GB")

Index             1010463072
id                 505231536
date              1010463072
store_nbr          126313165
item_nbr           253004826
unit_sales         505231536
is_onpromotion     126308108
is_holiday         126308108
transactions      1010463072
day               1010463072
month             1010463072
weekday           1010463072
year              1010463072
log_unit_sales     505231536
time_idx           505231536
dtype: int64
9.058138220570982 GB


In [17]:
# Prepare TimeSeriesDataSet
max_encoder_length = 30  # Using the past 30 days
max_prediction_length = 7  # Predicting the next 7 days
training_cutoff = data["time_idx"].max() - max_encoder_length - max_prediction_length


In [18]:
if 1 == -1:
    training = TimeSeriesDataSet(
    data=data[lambda x: x.time_idx <= training_cutoff],
    time_idx="time_idx",  # Use the integer index
    target="log_unit_sales",
    group_ids=["store_nbr", "item_nbr"],
    min_encoder_length=(max_encoder_length // 2)-1,
    max_encoder_length=max_encoder_length,
    static_categoricals=["store_nbr", "item_nbr"],
    time_varying_known_categoricals=["is_onpromotion", "is_holiday"],
    time_varying_unknown_categoricals=[],
    time_varying_known_reals=["day", "month", "weekday", "year", "transactions"],
    time_varying_unknown_reals=["log_unit_sales"],
    target_normalizer=GroupNormalizer(groups=["store_nbr", "item_nbr"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
    allow_missing_timesteps=True,
)


In [19]:
if 1 ==-1:
    # Save dataset object
    torch.save(training, "training_dataset.pt")



In [20]:
if 1==1:
    # Load dataset object
    training = torch.load("training_dataset.pt",weights_only=False)


In [21]:
if 1 == -1:
    # create validation set (predict=True) which means to predict the last max_prediction_length points in time for each series
    validation = TimeSeriesDataSet.from_dataset(
        training,
        data,
        predict=True,
        stop_randomization=True)

In [22]:
if 1 == -1:
# Save dataset object
    torch.save(validation, "validation_dataset.pt")


In [23]:
if 1 == 1:
    # Load dataset object
    validation = torch.load("validation_dataset.pt",weights_only=False)

In [24]:
# create dataloaders for model
batch_size = 128  # setting it between 32 to 128
train_dataloader = training.to_dataloader(train=True, batch_size=batch_size, num_workers=0,pin_memory=True)
val_dataloader = validation.to_dataloader(train=False, batch_size=batch_size , num_workers=0,pin_memory=True)

In [25]:
#Create baseline model

In [26]:
if 1==-1:
    # calculate baseline mean absolute error, i.e. predict next value as the last available value from the history
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    actuals = torch.cat([y for x, (y, weight) in iter(val_dataloader)]).to(device)
    baseline_predictions = Baseline().predict(val_dataloader)
    baseline_predictions = baseline_predictions.to(device)
    mae=(actuals - baseline_predictions).abs().mean().item()
    print(f"MAE loss for naive base model prediction on validation: {mae:.4f}")


In [27]:
if 1==-1:
    sm_loss = SMAPE().loss(actuals, baseline_predictions.to(device)).mean(axis = 1).median().item()
    print(f"Median loss for naive prediction on validation: {sm_loss:.4f}")


In [28]:
print(training.get_parameters())

{'time_idx': 'time_idx', 'target': 'log_unit_sales', 'group_ids': ['store_nbr', 'item_nbr'], 'weight': None, 'max_encoder_length': 30, 'min_encoder_length': 14, 'min_prediction_idx': 0, 'min_prediction_length': 1, 'max_prediction_length': 1, 'static_categoricals': ['store_nbr', 'item_nbr'], 'static_reals': None, 'time_varying_known_categoricals': ['is_onpromotion', 'is_holiday'], 'time_varying_known_reals': ['day', 'month', 'weekday', 'year', 'transactions'], 'time_varying_unknown_categoricals': [], 'time_varying_unknown_reals': ['log_unit_sales'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': True, 'lags': None, 'add_relative_time_idx': True, 'add_target_scales': True, 'add_encoder_length': True, 'target_normalizer': GroupNormalizer(
	method='standard',
	groups=['store_nbr', 'item_nbr'],
	center=True,
	scale_by_group=False,
	transformation='softplus',
	method_kwargs={}
), 'categorical_encoders': {'__group_id__store_nbr': NaNLabelEncoder(add_nan=Fa

In [29]:
print(validation.get_parameters())

{'time_idx': 'time_idx', 'target': 'log_unit_sales', 'group_ids': ['store_nbr', 'item_nbr'], 'weight': None, 'max_encoder_length': 30, 'min_encoder_length': 14, 'min_prediction_idx': 0, 'min_prediction_length': 1, 'max_prediction_length': 1, 'static_categoricals': ['store_nbr', 'item_nbr'], 'static_reals': None, 'time_varying_known_categoricals': ['is_onpromotion', 'is_holiday'], 'time_varying_known_reals': ['day', 'month', 'weekday', 'year', 'transactions'], 'time_varying_unknown_categoricals': [], 'time_varying_unknown_reals': ['log_unit_sales'], 'variable_groups': None, 'constant_fill_strategy': None, 'allow_missing_timesteps': True, 'lags': None, 'add_relative_time_idx': True, 'add_target_scales': True, 'add_encoder_length': True, 'target_normalizer': GroupNormalizer(
	method='standard',
	groups=['store_nbr', 'item_nbr'],
	center=True,
	scale_by_group=False,
	transformation='softplus',
	method_kwargs={}
), 'categorical_encoders': {'__group_id__store_nbr': NaNLabelEncoder(add_nan=Fa

In [30]:
#Train the Temporal Fusion Transformer

In [31]:
# optimizer = torch.optim.AdamW([
#     {"params": tft.add_norm_layer.parameters(), "lr": 1e-4},  # Lower LR for LayerNorm
#     {"params": tft.other_layers.parameters(), "lr": 1e-3}
# ], weight_decay=1e-5)


In [ ]:
if 1==1:
    tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.0005,
    hidden_size=128,              # Tuned hidden size
    attention_head_size=3,        # Tuned attention head size
    dropout=0.2,                 # Tuned dropout value
    hidden_continuous_size=40,    # Tuned continuous hidden size (<= hidden_size)
    loss=QuantileLoss(),
    optimizer="adamw",
    # output_size=7,  # Outputs for quantile forecasting
    logging_metrics=[MAE(), RMSE()],
    reduce_on_plateau_patience=1000,
)
print(f"Number of parameters in network: {tft.size() / 1e3:.1f}k")


In [ ]:
if 1 == -1:
    # configure network and trainer
    pl.seed_everything(69)
    # Define Trainer without any automatic LR finder callbacks (so tuner can be used)
    trainer = Trainer(
        devices=1,
        max_epochs=1,  # One epoch is enough for lr_find
        # gradient_clip_val=0.45,
        # accumulate_grad_batches=1,
        precision="bf16-mixed",
        gradient_clip_val=0.1,  # Helps prevent exploding gradients
        gradient_clip_algorithm="norm",  # Clip based on total norm
        accelerator="gpu" if torch.cuda.is_available() else "cpu",
        accumulate_grad_batches=8,
        limit_train_batches=1.0,  # Use 50% of the training batches
        limit_val_batches=1.0,    # Use 50% of the validation batches
        )

In [ ]:
if 1==-1:
    for name, param in tft.named_parameters():
        print(name, param.requires_grad)    



In [ ]:
if 1 == -1:
    # find optimal learning rate
    from lightning.pytorch.tuner import Tuner

    res = Tuner(trainer).lr_find(
        model=tft,
        train_dataloaders=train_dataloader,  # ensure train_dataloader is defined
        val_dataloaders=val_dataloader,        # ensure val_dataload`er is defined
        min_lr=1e-6,
        max_lr=1e-1,
        num_training=10000,  # number of batches to run for lr_find
        mode="exponential",
        early_stop_threshold=10.0,  # Stop early if loss gets too high
    )

In [ ]:
if 1==1:
    # Print and plot the suggested learning rate.
    suggested_lr = res.suggestion()
    print(f"Suggested learning rate: {suggested_lr}")
    fig = res.plot(show=True, suggest=True)
    fig.show()
    # Update the model's learning rate with the suggestion:
    tft.hparams.learning_rate = suggested_lr


In [ ]:
if 1==-1:
    from lightning.pytorch import LightningDataModule
    from pytorch_forecasting.data import TimeSeriesDataSet
    
    class TimeSeriesDataModule(LightningDataModule):
        def __init__(self, dataset: TimeSeriesDataSet, batch_size: int = 2):
            super().__init__()
            self.dataset = dataset
            self.batch_size = batch_size
    
        def train_dataloader(self):
            return self.dataset.to_dataloader(
                train=True,
                batch_size=self.batch_size,
                num_workers=0  # Adjust as needed
            )
    
        def val_dataloader(self):
            return self.dataset.to_dataloader(
                train=False,
                batch_size=self.batch_size,
                num_workers=0  # Adjust as needed
            )

In [ ]:
if 1==-1:
    datamodule = TimeSeriesDataModule(dataset=training, batch_size=2)



In [ ]:
if 1==-1:
    # Initialize the trainer and tuner
     trainer = Trainer(
            accelerator="cuda" if torch.cuda.is_available() else "cpu",
            devices=1,
            max_epochs=1,  # One epoch is enough for lr_find
            gradient_clip_val=0.45,
            accumulate_grad_batches=1,
            callbacks=[RichProgressBar()],
            precision="bf16-mixed"
            )
     tuner = Tuner(trainer)
        # Find the optimal batch size
     optimal_batch_size = tuner.scale_batch_size(
            model=tft,  # Your TFTModel instance
            datamodule=datamodule,
            method="fit",
            steps_per_trial=3,
            init_val=2,
            max_trials=25,
            batch_arg_name="batch_size"
        )
        
     print("Optimal batch size:", optimal_batch_size)

In [ ]:
tft.train()

In [ ]:
# tft.hparams.batch_size = 32  # Set initial batch size

In [ ]:
#callbacks
checkpoint_callback = ModelCheckpoint(
    dirpath="checkpoints/",  # Directory to save models
    filename="tft_epoch_{epoch:02d}",  # Naming format
    save_top_k=-1,  # Save all epochs
    every_n_epochs=1,  # Save after every epoch
)
lr_logger = LearningRateMonitor()  # log the learning rate
earlystopping_callback=EarlyStopping(
    monitor="val_loss",
     min_delta=1e-4,
    patience=5,verbose=False, mode="min")
logger = TensorBoardLogger("lightning_logs")  # logging results to a tensorboard


In [ ]:
# Set Up Trainer with GPU Acceleration
pl.seed_everything(69)
trainer = Trainer(
    max_epochs=50,  # Can increase if needed
    gradient_clip_val=0.1,  # Helps prevent exploding gradients
    gradient_clip_algorithm="norm",  # Clip based on total norm
    callbacks=[checkpoint_callback,earlystopping_callback,RichProgressBar()],
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1 if torch.cuda.is_available() else None,  # Use GPU if available
    precision="bf16-mixed",
    accumulate_grad_batches=8,
    profiler="simple",
    limit_train_batches=1.0,  # Use 50% of the training batches
    limit_val_batches=1.0,    # Use 50% of the validation batches
    log_every_n_steps=50  # Log less frequently
)

In [ ]:
print(trainer.callbacks)

Train model

In [ ]:
if 1==1:
    # fit network
    trainer.fit(
        tft,
        train_dataloaders=train_dataloader,
        val_dataloaders=val_dataloader,
    )


In [ ]:
# tft.save_checkpoint("TFT_v4.pth")